Pull Kobo Raw JSON and Filter ID 136

This notebook:
1. Connects to KoboToolbox API
2. Finds the target Kobo form
3. Pulls all submissions
4. Saves raw Kobo JSON
5. Filters for record ID 136


In [ ]:
import os
import json
import requests
from getpass import getpass
from pathlib import Path
import pandas as pd


In [ ]:
# Configuration
SERVER_URL = 'https://kf.kobotoolbox.org'
TARGET_FORM_NAME = 'gaia_metadata_authoring_form_v2'
TARGET_ID = '136'

OUTPUT_DIR = Path('kobo_output')
RAW_RESPONSE_FILE = OUTPUT_DIR / 'raw_kobo_response.json'
TARGET_RECORD_FILE = OUTPUT_DIR / f'record_{TARGET_ID}_raw.json'
FILTERED_CSV_FILE = OUTPUT_DIR / f'record_{TARGET_ID}_filtered.csv'


In [ ]:
# Authentication
API_TOKEN = (os.getenv('KOBO_API_TOKEN') or getpass('Paste Kobo API token: ')).strip()

headers = {
    'Authorization': f'Token {API_TOKEN}',
    'Accept': 'application/json',
}


In [ ]:
# Helper functions
def get_json(url, headers, timeout=60):
    response = requests.get(url, headers=headers, timeout=timeout)
    print(f'GET {response.url} -> {response.status_code}')
    response.raise_for_status()
    return response.json()


def fetch_paginated_results(start_url, headers):
    results = []
    next_url = start_url

    while next_url:
        payload = get_json(next_url, headers=headers)
        results.extend(payload.get('results', []))
        next_url = payload.get('next')

    return results


In [ ]:
# Fetch all Kobo assets/forms
assets_url = f'{SERVER_URL}/api/v2/assets/'
assets = fetch_paginated_results(assets_url, headers=headers)

print(f'Total assets found: {len(assets)}')


In [ ]:
# Find target form
target_asset = None
target_form_name_normalized = TARGET_FORM_NAME.strip().lower()

for asset in assets:
    asset_name = asset.get('name', '').strip().lower()

    if asset_name == target_form_name_normalized:
        target_asset = asset
        print(f"Success! Found form: {asset.get('name')}")
        print(f"UID: {asset.get('uid')}")
        break

if target_asset is None:
    raise ValueError(f"Could not find a form named {TARGET_FORM_NAME}")


In [ ]:
# Fetch all submissions
data_url = target_asset.get('data')

if not data_url:
    raise ValueError('The target Kobo form does not have a data endpoint.')

submissions = fetch_paginated_results(data_url, headers=headers)

print(f'Total submissions retrieved: {len(submissions)}')


In [ ]:
# Save raw Kobo JSON response
OUTPUT_DIR.mkdir(exist_ok=True)

raw_payload = {
    'form_name': target_asset.get('name'),
    'form_uid': target_asset.get('uid'),
    'count': len(submissions),
    'results': submissions,
}

with open(RAW_RESPONSE_FILE, 'w', encoding='utf-8') as f:
    json.dump(raw_payload, f, indent=2, ensure_ascii=False)

print(f'Saved raw response to: {RAW_RESPONSE_FILE}')


In [ ]:
# Convert to DataFrame for inspection
df = pd.json_normalize(submissions)

print('Available DataFrame columns:')
for col in df.columns:
    print('-', col)

print(f'DataFrame shape: {df.shape}')
display(df.head())


In [ ]:
# Filter for the specific legacy/control ID
id_column = 'control_group/id'

if id_column not in df.columns:
    raise KeyError(f"Could not find {id_column} in the DataFrame")

filtered_df = df[df[id_column].astype(str) == TARGET_ID]

print(f'Rows matching ID {TARGET_ID}: {len(filtered_df)}')
display(filtered_df)


In [ ]:
# Save filtered DataFrame and raw JSON record
filtered_df.to_csv(FILTERED_CSV_FILE, index=False)
print(f'Saved filtered CSV to: {FILTERED_CSV_FILE}')

record_136 = None

for record in submissions:
    if str(record.get(id_column)) == TARGET_ID:
        record_136 = record
        break

if record_136 is None:
    raise ValueError(f'No raw JSON record found for ID {TARGET_ID}')

with open(TARGET_RECORD_FILE, 'w', encoding='utf-8') as f:
    json.dump(record_136, f, indent=2, ensure_ascii=False)

print(f'Saved raw JSON record for ID {TARGET_ID} to: {TARGET_RECORD_FILE}')
